# Getting cutouts straight from the Zooniverse `.csv` export
The JSON files from Sophie's paper contain aggregated data from ~20000 volunteers.

We want to get at the "raw" box data. So we do that here.

A UMN researcher (Lestat) exported the Zooniverse "box the jet" information from the website to [a CSV file](https://drive.google.com/file/d/16A91wIPhhaEGvMGp0hrOeVNCsu-vwvVY/view?usp=sharing).
It contains information on the images which were boxed so that the bounding boxes can be converted from image to physical coordinates.
It was exported using `bash`:
```bash
# First, install the Zooniverse panoptes package
$(which uv) pip install panoptescli

# Configure it with username/password which has access to the SJH project
panoptes configure

# Finally, download the "box the jet" workflow to a CSV
panoptes workflow download-classifications 21225 box-the-jets.csv
# Optionally regenerate the workflow data set if required by adding the -g option
# panoptes workflow download-classifications -g 21225 box-the-jets.csv
```

There is also additional information regarding the "base" of each jet.
We don't care about the bases of the jets, just the bounding boxes, so we can ignore that.

In [ ]:
import json

import pandas as pd
import astropy.units as u

import zooniverse_processing as zp

In [ ]:
df = pd.read_csv("box-the-jets.csv")

# Exclude beta testing by restricting the version here
cutoff_version = 50
cut = df["workflow_version"] >= cutoff_version
df = df[cut]


In [ ]:
test_id = 88891958

In [ ]:
# Keep only the Zooniverse data we care about, and discard the rest
extracted = dict()

for i in range(df.shape[0]):
    row = df.iloc[i]
    id_ = int(row["subject_ids"])
    if id_ not in extracted.keys():
        extracted[id_] = {"bounding_boxes": list()}

    # The annotations row contains information on the bounding rectangles
    # contained in the current image set under investigation
    boxes = zp.extract_jethunter_annotations(raw_anno := json.loads(row["annotations"]))
    extracted[id_]["bounding_boxes"].extend(boxes)

    # The metadata of the current subject contains things like time,
    # image translation numbers, and FITS headers for physical coordinate conversion.
    # But, each subject ID will be visited by many volunteers, so only save the metadata
    # one time.
    if "meta" not in extracted[id_]:
        extracted[id_]["meta"] = zp.extract_jethunter_subject_data(
            raw_dat := json.loads(row["subject_data"])
        )
        only_key = int(list(raw_dat.keys())[0])
        if only_key == test_id:
            keep_dat = raw_dat[str(only_key)]
            keep_anno = raw_anno

In [ ]:
keep_anno

In [ ]:
cutoff = '2016-12-24'
time_good = dict()
for id_ in extracted:
    meta = extracted[id_]["meta"]
    if meta["time"]["start_time"] < cutoff:
        time_good[id_] = extracted[id_]
len(time_good), len(extracted)

In [ ]:
for id_, tg in time_good.items():
    if tg['meta']['time']['start_time'].startswith('2012-06-23'):
        print(id_, tg)

In [ ]:
# Test out converting the Zooniverse box centers into helioprojective arcseconds

test_id = 88891958
meta_ = time_good[test_id]["meta"]
bounding_boxes = extracted[test_id]["bounding_boxes"]
meta_

In [ ]:
from sunpy.net import attrs as a, Fido
email = "wilbert.steinbun@gmail.com"
t = a.Time(meta_['time']['start_time'], meta_['time']['end_time'])
query = Fido.search(
    t,
    a.Wavelength(304 << u.angstrom),
    a.jsoc.Series.aia_lev1_euv_12s,
    a.jsoc.Notify(email),
    a.jsoc.Segment.image,
    a.Sample(24 << u.s),
)
Fido.fetch(query, path='./example-data/', max_conn=10)

In [ ]:
# Test getting a sky region from the Zooniverse data
jet_regions = tuple(
    zp.sky_region_from_zooniverse_rect(box=b, meta=meta_)
    for b in bounding_boxes
)

In [ ]:
jet_regions

In [ ]:
import sunpy.map as smap
from sunpy.map.sources import sdo
from sunpy.coordinates import SphericalScreen
import matplotlib.pyplot as plt
%matplotlib qt
plt.style.use('../nice.mplstyle')

m: sdo.AIAMap = smap.Map('example-data/aia.lev1_euv_12s.2012-06-23T083022Z.304.image_lev1.fits')
fig, ax = plt.subplots(subplot_kw={'projection': m.wcs}, layout='tight')
m.plot(axes=ax)

with SphericalScreen(center=m.observer_coordinate):
    for jet_region in jet_regions:
        px = jet_region.to_pixel(wcs=m.wcs)
        px.plot(ax=ax, color='pink', alpha=0.1)
    # ax.scatter(*base_point.to(u.deg), color='blue', transform=ax.get_transform('world'))

plt.show()